# Experiment 1: Student Employability Classification
## Intelligent Multimodal Mock Interview Assessment System

### Objectives:
1. **Target Leakage Investigation**: Assess `Student Performance Rating` and other features for potential label leakage.
2. **Feature Selection**: Exclude appearance/physical features (`GENERAL APPEARANCE`, `PHYSICAL CONDITION`) and non-predictive identifiers (`Name of Student`).
3. **Feature Engineering**: Construct summary soft-skill interaction metrics (`soft_skill_mean`, `soft_skill_min`).
4. **Stratified Splitting**: 60% Train, 20% Validation, 20% Held-Out Test.
5. **Cross-Validation**: 5-Fold Stratified Cross-Validation on the training partition.
6. **Model Benchmarking**: Logistic Regression, Decision Tree, Random Forest, Support Vector Machine (SVM), Gradient Boosting, and K-Nearest Neighbors (KNN).
7. **Evaluation**: Accuracy, Precision, Recall, Macro/Binary F1, ROC-AUC, and Confusion Matrices.
8. **Model Serialization**: Export trained scikit-learn pipelines with Joblib.

In [ ]:
import os, json, joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# Paths
DATA_PATH = os.path.join("..", "dataset", "employability", "Student-Employability-Datasets.xlsx")
df = pd.read_excel(DATA_PATH, sheet_name="Data")
print(f"Raw dataset shape: {df.shape}")
df.head()

## 1. Target Leakage & Feature Selection Analysis

We investigate whether `Student Performance Rating` acts as target leakage or proxy for `CLASS`.

In [ ]:
df["CLASS_BIN"] = (df["CLASS"] == "Employable").astype(int)

print("=== Point-Biserial Correlation vs. Binary Target ===")
for col in df.columns:
    if col not in ["CLASS", "CLASS_BIN", "Name of Student"]:
        r, p = stats.pointbiserialr(df[col], df["CLASS_BIN"])
        print(f"{col:<32s} r={r:+.4f} (p={p:.2e})")

print("\nCrosstab: Student Performance Rating vs CLASS")
print(pd.crosstab(df["Student Performance Rating"], df["CLASS"], normalize='index').round(3))

**Findings on Leakage:**
- `Student Performance Rating` has $r = -0.0008$, and the class distribution is ~58% Employable across all ratings (3, 4, 5). It does not leak the target, but provides negligible discriminating signal.
- Excluded columns:
  1. `Name of Student` (identifier)
  2. `GENERAL APPEARANCE` (excluded per instructions: visual appearance)
  3. `PHYSICAL CONDITION` (excluded per instructions: physical condition)
  4. `Student Performance Rating` (excluded: non-discriminating)

## 2. Feature Engineering & Stratified Partitioning (60 / 20 / 20)

In [ ]:
features = [
    "MANNER OF SPEAKING", "MENTAL ALERTNESS", "SELF-CONFIDENCE",
    "ABILITY TO PRESENT IDEAS", "COMMUNICATION SKILLS"
]

X = df[features].copy()
X["soft_skill_mean"] = X[features].mean(axis=1)
X["soft_skill_min"] = X[features].min(axis=1)
y = df["CLASS"].copy()

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)

print(f"Train samples: {len(X_train)} ({len(X_train)/len(X):.1%})")
print(f"Val samples:   {len(X_val)} ({len(X_val)/len(X):.1%})")
print(f"Test samples:  {len(X_test)} ({len(X_test)/len(X):.1%})")

## 3. Pipeline Definitions & 5-Fold Stratified Cross-Validation

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", random_state=42))
    ]),
    "Decision Tree": Pipeline([
        ("clf", DecisionTreeClassifier(max_depth=6, min_samples_leaf=15, class_weight="balanced", random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("clf", RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=10, class_weight="balanced", random_state=42))
    ]),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42))
    ]),
    "Gradient Boosting": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8, random_state=42))
    ]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=9, weights="distance"))
    ])
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_records = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=["accuracy", "f1_macro"])
    cv_records.append({
        "Model": name,
        "CV Accuracy Mean": scores["test_accuracy"].mean(),
        "CV Accuracy Std": scores["test_accuracy"].std(),
        "CV F1 Mean": scores["test_f1_macro"].mean(),
        "CV F1 Std": scores["test_f1_macro"].std()
    })

pd.DataFrame(cv_records).sort_values("CV F1 Mean", ascending=False)

## 4. Final Evaluation on Held-Out Test Set

In [ ]:
test_records = []
for name, model in models.items():
    model.fit(X_trainval, y_trainval)
    y_pred = model.predict(X_test)
    emp_idx = list(model.classes_).index("Employable")
    y_prob = model.predict_proba(X_test)[:, emp_idx] if hasattr(model, "predict_proba") else None
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, pos_label="Employable")
    rec = recall_score(y_test, y_pred, pos_label="Employable")
    f1 = f1_score(y_test, y_pred, pos_label="Employable")
    auc = roc_auc_score((y_test == "Employable").astype(int), y_prob) if y_prob is not None else np.nan
    
    test_records.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(auc, 4)
    })

test_df = pd.DataFrame(test_records).sort_values("F1-Score", ascending=False)
test_df